# 02 · Base compartilhada (filtro de KPI)

**Projeto:** Cronos · Challenge FIAP 2026 com Locaweb
**Autores:** Ana Beatriz Costa de Oliveira · Hygor Abrantes · Igor Vignola
**Criado em:** 21/07/2026 · **Atualizado em:** 17/08/2026

Produção da base que os modelos reutilizam: carga do dataset, tipagem, aplicação do filtro
oficial de elegibilidade ao KPI e gravação em Parquet, com uma linha por incidente elegível.
Cada modelo remodela essa base conforme a necessidade: o Prophet agrega por dia, o modelo de
risco usa uma linha por incidente e o índice de saúde agrega por produto.

**Entrada:** `assets/Materal LocalWeb/LW-DATASET.xlsx`, aba `Dataset Geral`.
**Saída:** `data/interim/incidentes_kpi.parquet`.
**Bibliotecas:** pandas, openpyxl, pyarrow (Python 3.11+).

## 1. Setup

In [1]:
# Stdlib
from pathlib import Path

# Third-party
import pandas as pd

pd.set_option('display.max_columns', None)

# Resolve o caminho rodando da raiz do repo ou de notebooks/.
candidatos = [Path('assets/Materal LocalWeb/LW-DATASET.xlsx'),
              Path('../assets/Materal LocalWeb/LW-DATASET.xlsx')]
XLSX = next(p for p in candidatos if p.exists())
REPO = XLSX.parents[2]
INTERIM = REPO / 'data' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

print('dataset:', XLSX)
print('saída:  ', INTERIM / 'incidentes_kpi.parquet')

dataset: ..\assets\Materal LocalWeb\LW-DATASET.xlsx
saída:   ..\data\interim\incidentes_kpi.parquet


## 2. Carga dos dados

Leitura da aba `Dataset Geral`, sem transformação. O `assert` fixa o número de linhas do dataset
oficial: se a Locaweb entregar outra versão do arquivo, todo número a jusante muda, e é melhor
falhar aqui do que propagar em silêncio.

In [2]:
df_raw = pd.read_excel(XLSX, sheet_name='Dataset Geral')

# Invariante do dataset oficial da Locaweb; se mudar, todos os numeros a jusante mudam.
assert len(df_raw) == 122_543, f'Esperadas 122.543 linhas no dataset bruto, lidas {len(df_raw):,}'

print(f'Linhas: {len(df_raw):,} | Colunas: {df_raw.shape[1]}')
df_raw.head(3)

Linhas: 122,543 | Colunas: 19


,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,Duração,Código de fechamento,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?
0,INC8654273,3 - Média,NaN,NaN,NaN,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,14,NaN,Problem: Apache Busy Workers,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
1,INC8654270,4 - Baixa,NaN,NaN,NaN,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,209,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
2,INC8654264,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,110,NaN,Problem: Alarm Application Monitoring database...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN


## 3. Tipagem

Conversão das três colunas de data para `datetime` e de `Duração` para numérico, ambas com
`errors='coerce'`. A diferença de nulos antes e depois mede quantos valores a conversão não
conseguiu interpretar. `Aberto` não admite nulo, porque alimenta toda a série temporal.

In [3]:
cols_convertidas = ['Aberto', 'Resolvido', 'Encerrado', 'Duração']
nulos_antes = df_raw[cols_convertidas].isna().sum()

for col in ['Aberto', 'Resolvido', 'Encerrado']:
    df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce')
df_raw['Duração'] = pd.to_numeric(df_raw['Duração'], errors='coerce')

print(df_raw[cols_convertidas].dtypes)
print("\nNulos gerados pela conversão (errors='coerce'):")
print((df_raw[cols_convertidas].isna().sum() - nulos_antes).to_string())

# 'Aberto' e a coluna que alimenta toda a serie temporal a jusante.
assert df_raw['Aberto'].notna().all(), 'Coluna Aberto contém nulos após a conversão; a série temporal depende dela.'

Aberto       datetime64[ns]
Resolvido    datetime64[ns]
Encerrado    datetime64[ns]
Duração               int64
dtype: object

Nulos gerados pela conversão (errors='coerce'):
Aberto       0
Resolvido    0
Encerrado    0
Duração      0


A conversão não gerou nulo em nenhuma das quatro colunas: o Excel já entrega as datas tipadas
como `datetime64` e `Duração` como inteiro. A tipagem é redundante para este arquivo e fica como
proteção contra uma exportação futura em texto.

## 4. Filtro de elegibilidade ao KPI

O recorte usa o campo oficial `Entrou para KPI? == 'SIM'`. O dicionário de dados descreve a mesma
elegibilidade por uma regra composta: incidente pai vazio, prioridade 1/2/3 e status diferente de
`Sem Intervenção`. As duas leituras são comparadas abaixo.

In [4]:
df_kpi = df_raw[df_raw['Entrou para KPI?'] == 'SIM'].copy()

# Invariante do filtro oficial; número citado em documentos e slides do projeto.
assert len(df_kpi) == 25_600, f'Esperados 25.600 incidentes elegíveis ao KPI, obtidos {len(df_kpi):,}'

print(f'total:      {len(df_raw):,}')
print(f'elegíveis:  {len(df_kpi):,}  ({len(df_kpi) / len(df_raw) * 100:.0f}%)')
print(f'período:    {df_kpi["Aberto"].min().date()} a {df_kpi["Aberto"].max().date()}')

# Recorte que já foi confundido com elegibilidade e devolve outro universo.
print(f'\nsó com Incidente Pai vazio: {int(df_raw["Incidente Pai"].isna().sum()):,} '
      f'({df_raw["Incidente Pai"].isna().mean() * 100:.0f}% da base)')

total:      122,543
elegíveis:  25,600  (21%)
período:    2023-01-02 a 2025-12-31

só com Incidente Pai vazio: 107,416 (88% da base)


In [5]:
# Regra composta descrita no dicionário de dados, para comparar com o campo oficial.
regra_dicionario = (df_raw['Incidente Pai'].isna()
                    & df_raw['Prioridade'].astype(str).str[0].isin(['1', '2', '3'])
                    & (df_raw['Status'] != 'Sem Intervenção'))
campo_oficial = df_raw['Entrou para KPI?'] == 'SIM'

print(pd.crosstab(regra_dicionario, campo_oficial,
                  rownames=['regra do dicionário'], colnames=['campo oficial']))

so_regra = int((regra_dicionario & ~campo_oficial).sum())
so_campo = int((~regra_dicionario & campo_oficial).sum())
print(f'\nconcordância: {(regra_dicionario == campo_oficial).mean() * 100:.2f}%')
print(f'regra inclui e campo exclui: {so_regra}')
print(f'campo inclui e regra exclui: {so_campo}')

campo oficial        False  True 
regra do dicionário              
False                96792      0
True                   151  25600

concordância: 99.88%
regra inclui e campo exclui: 151
campo inclui e regra exclui: 0


O campo oficial concorda com a regra do dicionário em 99,88% dos registros. As 151 divergências
caem todas no mesmo sentido: a regra composta incluiria incidentes que o campo exclui, e não há
nenhum caso no sentido inverso. **O campo oficial é o critério mais restritivo, e é o que a
Locaweb usa para apurar o KPI**, então é ele que define o recorte.

Filtrar apenas por `Incidente Pai` vazio devolve 107.416 registros, 88% da base. É um recorte
diferente de elegibilidade ao KPI.

## 5. Colunas de calendário e concentração temporal

Cinco colunas derivadas de `Aberto` e leitura da série diária de 2025 que os modelos de volume
consomem.

In [6]:
df_kpi['dia'] = df_kpi['Aberto'].dt.normalize()
df_kpi['dia_semana'] = df_kpi['Aberto'].dt.dayofweek   # 0 = segunda
df_kpi['hora'] = df_kpi['Aberto'].dt.hour
df_kpi['ano'] = df_kpi['Aberto'].dt.year
df_kpi['mes'] = df_kpi['Aberto'].dt.month

por_ano = df_kpi['ano'].value_counts().sort_index()
print('elegíveis por ano:')
for ano, n in por_ano.items():
    print(f'  {ano}: {n:,} ({n / len(df_kpi) * 100:.1f}%)'.replace(',', '.'))
print(f'  2023 e 2024 somam {int(por_ano.reindex([2023, 2024]).sum())} registros')

print('\nprioridade na base completa:')
print(df_raw['Prioridade'].value_counts().to_string())
print('\nprioridade entre os elegíveis:')
print(df_kpi['Prioridade'].value_counts().to_string())

# A série é reindexada no calendário de 2025 para que dia sem incidente entre como zero em vez
# de sumir, o que faria a média ser calculada sobre um denominador menor que 365.
# 'Prioridade' vem como texto no formato 'N - Nome'; o prefixo numérico identifica o nível.
calendario_2025 = pd.date_range('2025-01-01', '2025-12-31', freq='D')
df_2025 = df_kpi[df_kpi['ano'] == 2025]

print('\nsérie diária de 2025:')
for tag, prefixo in [('P2', '2'), ('P3', '3')]:
    observado = df_2025[df_2025['Prioridade'].astype(str).str.startswith(prefixo)].groupby('dia').size()
    serie = observado.reindex(calendario_2025, fill_value=0)
    print(f'  {tag}: média {serie.mean():.1f}/dia | mín {serie.min()} | máx {serie.max()}'
          f' | dias com incidente {len(observado)}/{len(calendario_2025)}')

elegíveis por ano:
  2023: 87 (0.3%)
  2024: 357 (1.4%)
  2025: 25.156 (98.3%)
  2023 e 2024 somam 444 registros

prioridade na base completa:
Prioridade
4 - Baixa          64828
3 - Média          41732
2 - Alta           15649
5 - Muito Baixa      333
1 - Crítica            1

prioridade entre os elegíveis:
Prioridade
3 - Média    20441
2 - Alta      5159

série diária de 2025:
  P2: média 14.1/dia | mín 3 | máx 43 | dias com incidente 365/365
  P3: média 54.8/dia | mín 1 | máx 168 | dias com incidente 365/365


98,3% dos elegíveis estão em 2025; 2023 e 2024 somam 444 registros. **O treino dos modelos de
volume usa 2025**, com sazonalidade semanal e sem sazonalidade anual, porque um ano de histórico
não permite estimar ciclo anual.

A série cobre os 365 dias do ano nas duas prioridades, então a reindexação não altera as médias
(P2 14,1/dia, P3 54,8/dia). Ela fica no código porque a média sobre dias observados passaria a
divergir da média sobre o calendário no primeiro dia sem incidente.

P1 tem 1 registro em toda a base e nenhum entre os elegíveis, o que não forma série. P4 e P5 não
entram no KPI. O escopo de previsão de volume é P2 e P3.

## 6. Escrita da base

Gravação em Parquet e releitura do arquivo para conferir que shape e tipos sobreviveram à
serialização.

In [7]:
print('schema (19 colunas originais + 5 de calendário):')
print(df_kpi.dtypes.to_string())

out = INTERIM / 'incidentes_kpi.parquet'
df_kpi.to_parquet(out, index=False)

conferencia = pd.read_parquet(out)
assert conferencia.shape == df_kpi.shape, 'shape mudou na ida e volta do parquet'
assert (conferencia.dtypes == df_kpi.dtypes).all(), 'tipo de coluna mudou na ida e volta do parquet'

print(f'\nsalvo: {out}')
print(f'relido: {conferencia.shape[0]:,} linhas x {conferencia.shape[1]} colunas, tipos preservados')

schema (19 colunas originais + 5 de calendário):
Número                          object
Prioridade                      object
Produto                         object
Categoria                       object
Subcategoria                    object
Grupo designado                 object
Item de configuração            object
Aberto                  datetime64[ns]
Resolvido               datetime64[ns]
Encerrado               datetime64[ns]
Duração                          int64
Código de fechamento            object
Descrição resumida              object
Solução                         object
Aberto por                      object
Incidente Pai                   object
Status                          object
Entrou para KPI?                object
KPI Violado?                    object
dia                     datetime64[ns]
dia_semana                       int32
hora                             int32
ano                              int32
mes                              int32



salvo: ..\data\interim\incidentes_kpi.parquet
relido: 25,600 linhas x 24 colunas, tipos preservados


## 7. Conclusão

A base gravada tem uma linha por incidente elegível ao KPI, tipada e com colunas de calendário.
98,3% dos registros estão em 2025, o que define o período de treino dos modelos de volume e a
decisão de manter apenas a sazonalidade semanal. O escopo de prioridade é P2 e P3.